In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

In [9]:
def impute_missing(df, strategy="median"):

    df_imputed = df.copy()
    numerical_cols = df_imputed.select_dtypes(include=[np.number]).columns

    for col in numerical_cols:
        if df_imputed[col].isnull().sum() > 0:
            if strategy == "median":
              imputed_value = df_imputed[col].median()
              df_imputed[col].fillna(imputed_value, inplace=True)
            elif strategy == "mean":
              imputed_value = df_imputed[col].mean()
              df_imputed[col].fillna(imputed_value, inplace=True)
            else:
                raise ValueError("strategy must be 'median' or 'mean'")

            print(f"  Imputed {col} with {strategy}: {df_imputed[col].median():.2f}")

    print(f"Imputation complete. Remaining nulls: {df_imputed.isnull().sum().sum()}")
    return df_imputed

In [16]:
def encode_categoricals(df):

    categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

    if not categorical_cols:
        print("No categorical columns found - skipping encoding.")
        return df

    print("-" * 40)
    print(" One-Hot Encoding")
    print("-" * 40)
    print(f"Columns to encode: {categorical_cols}")

    for col in categorical_cols:
        print(f"\n{col}:")
        print(f"  Unique values: {df[col].unique()}")
        print(f"  Value counts:\n{df[col].value_counts()}")

    df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=False)

    new_cols = [col for col in df_encoded.columns if any(col.startswith(f"{c}_") for c in categorical_cols)]
    print(f"\nNew columns created: {new_cols}")
    print(f"Shape before encoding: {df.shape} | Shape after: {df_encoded.shape}")
    print("-" * 40)

    return df_encoded

In [11]:
def split_data(df, target_col, test_size=0.2, random_state=42):

    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    print(f"Split complete:")
    print(f"  X_train: {X_train.shape} | X_test: {X_test.shape}")
    print(f"  y_train class balance: {y_train.value_counts(normalize=True).round(3).to_dict()}")
    print(f"  y_test  class balance: {y_test.value_counts(normalize=True).round(3).to_dict()}")
    return X_train, X_test, y_train, y_test

In [12]:
def apply_smote(X_train, y_train, random_state=42):

    print(f"Before SMOTE - class distribution: {pd.Series(y_train).value_counts().to_dict()}")

    smote = SMOTE(random_state=random_state)
    X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

    print(f"After SMOTE  - class distribution: {pd.Series(y_resampled).value_counts().to_dict()}")
    print(f"Training set size: {X_train.shape[0]} -> {X_resampled.shape[0]}")
    return X_resampled, y_resampled

In [13]:
print("-" * 50)
print(" All preprocessing functions loaded successfully")
print("-" * 50)
print("  impute_missing(df, strategy='median')")
print("  encode_categoricals(df)")
print("  split_data(df, target_col, test_size=0.2)")
print("  apply_smote(X_train, y_train)")
print("-" * 50)
print("\nReady to use in datasets!")

--------------------------------------------------
 All preprocessing functions loaded successfully
--------------------------------------------------
  impute_missing(df, strategy='median')
  encode_categoricals(df)
  split_data(df, target_col, test_size=0.2)
  apply_smote(X_train, y_train)
--------------------------------------------------

Ready to use in datasets!
